# Guía de Consulta Rápida — Visión Artificial (P1 y P2)

Este notebook está diseñado como **referencia rápida para el examen parcial**. Contiene:
1. **Bloque de Imports Universal** (copiar y pegar al empezar).
2. **Buscador rápido: «¿Qué quiero hacer? → Código exacto»**.
3. **Tabla de Tipos de Datos, Rangos y Conversiones (`uint8` vs `float`)**.
4. **Recetario P1:** Creación, Slicing, Marcos, Brillo, Contraste (`rescale_intensity`), Canales RGB/RGBA y Máscaras Booleanas.
5. **Recetario P2:** Histogramas, Reescalado (`resize`/`rescale`), Convolución 1D/2D/RGB, Filtros (Gaussiano vs Mediana), Bordes (Prewitt, Sobel, Canny + Manual), Imágenes Híbridas y Anonimización ROI.
6. **Preguntas Teóricas Clave Resueltas** (listas para responder en el examen).

> **Guías temáticas detalladas por práctica:**
> - `Apuntes_P1.ipynb`
> - `Apuntes_P2.ipynb`

---
## 1. Bloque de Imports Universal (Copiar y Pegar)
Ejecuta esta celda al principio de cualquier notebook para tener todas las funciones de **P1** y **P2** disponibles:

In [1]:
%matplotlib inline
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from matplotlib import pyplot as plt

# Lectura, escritura y utilidades de tipos de datos
from skimage import io, exposure, filters, morphology, img_as_float, img_as_ubyte
from skimage.color import rgb2gray
from skimage.exposure import rescale_intensity
from skimage.transform import resize, rescale

# Filtrado, morfología y bordes (P2)
from scipy.ndimage import convolve
from skimage.filters import gaussian, median, sobel, prewitt
from skimage.morphology import disk, rectangle, diamond
from skimage.feature import canny

---
## 2. Buscador Rápido: «¿Qué quiero hacer? → Función exacta»

| Operación | Código / Función | Detalles y Trampas de Examen |
| :--- | :--- | :--- |
| **Crear imagen negra 2D (Gris)** | `img = np.zeros((H, W), dtype=np.uint8)` | Rango `[0, 255]`, 1 canal `(H, W)` |
| **Crear imagen negra 3D (RGB)** | `img = np.zeros((H, W, 3), dtype=np.uint8)` | 3 canales `(H, W, 3)` |
| **Crear imagen blanca** | `img = 255 * np.ones((H, W), dtype=np.uint8)` | Si es `float`, usar `np.ones((H, W))` |
| **Leer / Guardar imagen** | `img = io.imread('ruta.png')` / `io.imsave('out.png', img)` | Antes de `imsave`, convertir a `uint8` con `img_as_ubyte(img)` |
| **Convertir RGB a Gris** | `gray = rgb2gray(img_rgb[:, :, :3])` | **Cambia `dtype` a `float64` en `[0.0, 1.0]`** y quita el canal 3 |
| **Convertir a `uint8` `[0, 255]`** | `img_u8 = img_as_ubyte(img_float)` | Requiere que el `float` esté en `[-1, 1]` o `[0, 1]` |
| **Convertir a `float` `[0, 1]`** | `img_fl = img_as_float(img_u8)` | También sirve `img.astype(float) / 255` |
| **Mostrar sin auto-contraste** | `plt.imshow(img, cmap='gray', vmin=0, vmax=255)` | Si `img` es `float` `[0, 1]`, usar `vmin=0, vmax=1` |
| **Voltear verticalmente (Espejo)**| `np.flipud(img)` o `img[::-1, :]` o `np.flip(img, axis=0)` | Horizontal sería `np.fliplr(img)` o `img[:, ::-1]` |
| **Binarizar por umbral (media)** | `binary = img_gray > np.mean(img_gray)` | Devuelve array `bool` (`True`=1 blanco, `False`=0 negro) |
| **Contrast Stretching (Fórmula)**| `norm = (img - img.min()) / (img.max() - img.min())` | Devuelve `float64` en `[0.0, 1.0]` → usar `vmin=0, vmax=1` |
| **Contrast Stretching (skimage)**| `resc = rescale_intensity(img)` | Mantiene el `dtype` original (si era `uint8`, estira a `[0, 255]`) |
| **Intercambiar canales R y B** | `swapped = img.copy(); swapped[:,:,0] = img[:,:,2]; swapped[:,:,2] = img[:,:,0]` | Usar siempre `.copy()` para no sobreescribir el canal original |
| **Máscara píxeles no negros** | `mask = np.any(img[:, :, :3] > 0, axis=2)` | Ignora el canal Alpha (`:3`). No usar bucles `for` |
| **Histograma (Matplotlib)** | `ax.hist(img[:, :, 0].ravel(), bins=nbins)` | `.ravel()` aplana la matriz 2D a un vector 1D |
| **Histograma (Scikit-Image)** | `hist, bins = exposure.histogram(img, nbins=nbins)` | Se grafica con `ax.bar(bins, hist, width=bins[1]-bins[0])` |
| **Cambiar tamaño fijo `(H, W)`** | `img_res = resize(img, (256, 256))` | Devuelve `float64` en `[0.0, 1.0]` |
| **Escalar por factor (`1/20`, `20`)**| `img_sc = rescale(img, 1/20)` | Para RGB en versiones nuevas puede requerir `channel_axis=-1` |
| **Convolución 2D (Gris)** | `mask = mask / np.sum(mask); out = convolve(img, mask)` | Normalizar máscara de suavizado para que sume 1 |
| **Filtro Gaussiano** | `out = gaussian(img, sigma=1)` | Mayor `sigma` = más desenfoque (elimina altas frecuencias) |
| **Filtro de Mediana** | `out = median(img, morphology.disk(5))` | No lineal, preserva mejor bordes; formas: `disk(r)`, `rectangle(h,w)`, `diamond(r)` |
| **Bordes (Prewitt / Sobel / Canny)**| `prewitt(gray)`, `sobel(gray)`, `canny(gray, sigma=1)` | Prewitt/Sobel devuelven magnitud (`float`); Canny devuelve binario (`bool`) |
| **Frecuencias e Imagen Híbrida** | `low = gaussian(I, 4); high = I - low; hyb = 2*(0.5*low1 + 0.5*high2)` | Baja frecuencia = suavizada; Alta frecuencia = original − baja |

---
## 3. Tipos de Datos (`dtype`), Niveles de Intensidad y `vmin`/`vmax`

### ¿Cuántos niveles de intensidad tiene una imagen `uint8`?
- `uint8` utiliza **8 bits** por píxel (sin signo). Cada bit puede valer `0` o `1`.
- Por tanto, puede representar $2^8 = 256$ niveles distintos de intensidad, desde **`0` (negro)** hasta **`255` (blanco)**.

### Tabla de Tipos de Datos en `skimage` y `numpy`
| Tipo de Imagen / Función | `dtype` resultante | Rango de valores | Dimensiones (`shape`) | Parámetros `plt.imshow` |
| :--- | :--- | :--- | :--- | :--- |
| Binaria (`img > umbral`, `canny`) | `bool` | `False` (`0`) o `True` (`1`) | `(H, W)` | `cmap='gray'` |
| Escala de grises 8-bit | `uint8` | `0` a `255` | `(H, W)` | `cmap='gray', vmin=0, vmax=255` |
| Color RGB 8-bit | `uint8` | `0` a `255` por canal | `(H, W, 3)` | `plt.imshow(img)` |
| Color RGBA (con transparencia) | `uint8` | `0` a `255` por canal | `(H, W, 4)` | `plt.imshow(img)` |
| Salida de `rgb2gray`, `resize`, `rescale`, `img_as_float`, `gaussian`, Fórmula `[0,1]` | `float64` | `0.0` a `1.0` | `(H, W)` o `(H, W, 3)` | `cmap='gray', vmin=0, vmax=1` |


> **Trampa clásica de `plt.imshow()`:** Por defecto, `imshow()` estira automáticamente el mínimo de la imagen al negro y el máximo al blanco. Si restas o sumas brillo a una imagen de bajo contraste y **no** pones `vmin=0, vmax=255` (o `vmin=0, vmax=1` si es `float`), **se verá exactamente igual que la original**.

---
## 4. Recetario de Código — Práctica 1 (Manipulación Básica y Color)

### 4.1 Crear imágenes por regiones (Slicing) y combinarlas en canales RGB

In [2]:
# Base negra 100x100 uint8
img = np.zeros((100, 100), dtype=np.uint8)

# Mitad derecha blanca, mitad inferior blanca, cuadrante sup-izq blanco
img1 = img.copy(); img1[:, 50:] = 255
img2 = img.copy(); img2[50:, :] = 255
img3 = img.copy(); img3[:50, :50] = 255

# Volteo vertical (3 formas equivalentes)
img_flipped = np.flipud(img3)          # Opción 1
img_flipped = img3[::-1, :]            # Opción 2 (slicing)
img_flipped = np.flip(img3, axis=0)    # Opción 3

# Combinar 3 imágenes 2D en una imagen RGB 3D (100x100x3)
img_color = np.zeros((100, 100, 3), dtype=np.uint8)
img_color[:, :, 0] = img1  # Canal Rojo (R)
img_color[:, :, 1] = img2  # Canal Verde (G)
img_color[:, :, 2] = img3  # Canal Azul (B)

### 4.2 Añadir un marco (centrar imagen en un lienzo X% más grande) y colorear el marco

In [3]:
def enmarcar_imagen(img_rgb, factor=1.15, color_marco=(0, 255, 0)):
    height, width, channels = img_rgb.shape
    new_height = int(height * factor)
    new_width = int(width * factor)

    # 1. Crear lienzo negro
    framed = np.zeros((new_height, new_width, 3), dtype=np.uint8)

    # 2. Calcular índices de inicio para centrar (división entera //)
    start_row = (new_height - height) // 2
    start_col = (new_width - width) // 2

    # 3. Incrustar imagen original en el centro
    framed[start_row:start_row + height, start_col:start_col + width] = img_rgb

    # 4. (Opcional) Pintar los 4 bordes del marco de un color RGB
    framed[:start_row, :] = color_marco               # Arriba
    framed[start_row + height:, :] = color_marco      # Abajo
    framed[:, :start_col] = color_marco               # Izquierda
    framed[:, start_col + width:] = color_marco       # Derecha
    return framed

### 4.3 Ajuste de Brillo y Estiramiento de Contraste (Contrast Stretching)
$$I_{\text{normalized}} = \frac{I - I_{\min}}{I_{\max} - I_{\min}}$$

In [4]:
# Oscurecer para que el mínimo sea 0
dark_image = img_cell - img_cell.min()

# Aclarar para que el máximo sea 255
offset = 255 - img_cell.max()
bright_im = img_cell + offset

# Contrast Stretching Opción 1: Fórmula matemática -> devuelve float64 en [0.0, 1.0]
img_formula = (img_cell - img_cell.min()) / (img_cell.max() - img_cell.min())
# Visualizar con: plt.imshow(img_formula, cmap='gray', vmin=0, vmax=1)

# Contrast Stretching Opción 2: rescale_intensity -> mantiene uint8 y estira a [0, 255]
img_rescale = rescale_intensity(img_cell)
# Visualizar con: plt.imshow(img_rescale, cmap='gray', vmin=0, vmax=255)

NameError: name 'img_cell' is not defined

### 4.4 Máscaras Booleanas y Superposición en Imágenes RGBA (Sin bucles `for`)

In [ ]:
# Detectar píxeles NO negros considerando solo los canales RGB (:3) e ignorando Alpha
mask = np.any(img_coat[:, :, :3] > 0, axis=2)  # También válido: != 0

# Intercambiar canales Rojo (0) y Azul (2) solo en la máscara
img_coat_changed = img_coat.copy()
img_coat_changed[mask, 0] = img_coat[mask, 2]
img_coat_changed[mask, 2] = img_coat[mask, 0]

# Superponer el abrigo sobre el modelo modificando solo RGB (:3) y respetando Alpha
img_model_coat = img_model.copy()
img_model_coat[mask, :3] = img_coat_changed[mask, :3]

---
## 5. Recetario de Código — Práctica 2 (Histogramas, Filtrado y Bordes)

### 5.1 Histogramas RGB y Escala de Grises (2 Métodos)

In [ ]:
def rgb_im_hist_plt(img, nbins=8):
    """Método 1: Usando plt.hist() con .ravel()"""
    if not 2 <= nbins <= 256:
        raise ValueError("nbins must be between 2 and 256.")
    img_gray = rgb2gray(img)
    fig, ax = plt.subplots(1, 5, figsize=(20, 5))
    ax[0].imshow(img); ax[0].axis("off"); ax[0].set_title("Original Image")
    for i, name in enumerate(["Red Channel", "Green Channel", "Blue Channel"]):
        ax[i+1].hist(img[:, :, i].ravel(), bins=nbins)
        ax[i+1].set_title(name); ax[i+1].set_xlabel("Intensity"); ax[i+1].set_ylabel("Frequency")
    ax[4].hist(img_gray.ravel(), bins=nbins)
    ax[4].set_title("Grayscale"); ax[4].set_xlabel("Intensity"); ax[4].set_ylabel("Frequency")
    plt.tight_layout(); plt.show()

def rgb_im_hist_skimage(img, nbins=8):
    """Método 2: Usando exposure.histogram() + plt.bar()"""
    if not 2 <= nbins <= 256:
        raise ValueError("nbins must be between 2 and 256.")
    img_gray = rgb2gray(img)
    fig, ax = plt.subplots(1, 5, figsize=(20, 5))
    ax[0].imshow(img); ax[0].axis("off"); ax[0].set_title("Original Image")
    channels = [img[:, :, 0], img[:, :, 1], img[:, :, 2], img_gray]
    titles = ["Red Channel", "Green Channel", "Blue Channel", "Grayscale"]
    for i, (ch, title) in enumerate(zip(channels, titles)):
        hist, bins = exposure.histogram(ch, nbins=nbins)
        ax[i+1].bar(bins, hist, width=bins[1] - bins[0])
        ax[i+1].set_title(title); ax[i+1].set_xlabel("Intensity"); ax[i+1].set_ylabel("Frequency")
    plt.tight_layout(); plt.show()

### 5.2 Convolución 1D, 2D y en Imágenes RGB (`conv_coor`)

In [ ]:
# 1. Máscara 1D Horizontal (1x15): ¡Debe ser 2D [[1, ..., 1]] para scipy.ndimage.convolve!
mask_h1d = np.ones((1, 15))
mask_h1d = mask_h1d / np.sum(mask_h1d)  # Normalizar para que sume 1
img_h1d = convolve(oryx_gray, mask_h1d)

# 2. Máscara 1D Vertical (15x1): Transpuesta de la horizontal
mask_v1d = mask_h1d.T
img_v1d = convolve(oryx_gray, mask_v1d)

# 3. Máscara 2D Uniforme (15x15)
mask_2d = np.ones((15, 15))
mask_2d = mask_2d / np.sum(mask_2d)
img_2d = convolve(oryx_gray, mask_2d)

# 4. Convolución 2D aplicada canal por canal a una imagen RGB
def conv_coor(image, mask):
    result_image = np.zeros_like(image, dtype=float)
    mask = mask / np.sum(mask)  # Normalizar máscara
    for i in range(3):
        result_image[:, :, i] = convolve(image[:, :, i], mask)
    return result_image

### 5.3 Filtros Gaussiano vs Mediana y Detectores de Bordes (Integrados y Manuales)

In [ ]:
# --- FILTROS DE SUAVIZADO ---
g_smooth = gaussian(oryx_gray_float, sigma=1)                # Ponderado (Gaussiano)
m_disk   = median(oryx_gray_float, morphology.disk(5))       # Mediana circular (radio=5)
m_rect   = median(oryx_gray_float, morphology.rectangle(5, 20)) # Mediana rectangular (alto=5, ancho=20)
m_diam   = median(oryx_gray_float, morphology.diamond(3))    # Mediana rombo (radio=3)

# --- DETECTORES DE BORDES INTEGRADOS (skimage) ---
edges_prewitt = prewitt(logo_gray)            # Magnitud del gradiente (gris)
edges_sobel   = sobel(logo_gray)              # Magnitud del gradiente ponderado (gris)
edges_canny   = canny(logo_gray, sigma=1)     # Bordes finos umbralizados (binaria)

# --- DETECTOR DE BORDES MANUAL (Prewitt y Sobel con convolve) ---
sobel_x = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
sobel_y = np.array([[1, 2, 1],  [0, 0, 0],  [-1, -2, -1]])

prewitt_x = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
prewitt_y = np.array([[1, 1, 1],  [0, 0, 0],  [-1, -1, -1]])

def detect_edges(img, filter_x, filter_y, filters_normalized=False):
    float_img = img_as_float(img)
    gx = convolve(float_img, filter_x)  # Bordes verticales
    gy = convolve(float_img, filter_y)  # Bordes horizontales
    edges_img = np.sqrt(gx**2 + gy**2)  # Magnitud del gradiente
    if not filters_normalized:
        edges_img = (edges_img - np.min(edges_img)) / (np.max(edges_img) - np.min(edges_img))
    return img_as_ubyte(edges_img)

### 5.4 Imágenes Híbridas y Anonimización de Regiones (ROI)

In [ ]:
# --- IMÁGENES HÍBRIDAS ---
# 1. Igualar tamaño
img1 = resize(img1, (300, 300))
img2 = resize(img2, (300, 300))

# 2. Paso bajo L(I) con filtro Gaussiano y Paso alto H(I) = I - L(I)
img1_low = gaussian(img1, sigma=4)
img1_high = img1 - img1_low

img2_low = gaussian(img2, sigma=4)
img2_high = img2 - img2_low

# 3. Combinar frecuencias bajas de una con altas de otra
hybrid = 2 * (0.5 * img1_low + 0.5 * img2_high)

# --- ANONIMIZACIÓN DE REGIÓN RECTANGULAR (MATRÍCULAS / CARAS) ---
def anonymize_region(image_float, top_left, bottom_right, kernel_size=25):
    r1, c1 = top_left
    r2, c2 = bottom_right
    region = image_float[r1:r2, c1:c2]
    mask = np.ones((kernel_size, kernel_size))
    image_float[r1:r2, c1:c2] = conv_coor(region, mask)

---
## 6. Preguntas Teóricas de Examen y Respuestas Directas

1. **¿Por qué hay que normalizar la máscara en los filtros de suavizado pero NO en los detectores de bordes (Prewitt/Sobel)?**
   - **Suavizado (paso bajo):** Calcula un promedio local de intensidades. Si la suma de los pesos de la máscara no es `1`, el brillo medio de la imagen cambiará (se saturará o cambiará de escala).
   - **Bordes (Prewitt/Sobel):** Calculan **derivadas / gradientes** (diferencias entre píxeles vecinos). La suma de los elementos de un kernel derivativo es `0` (para que en zonas de intensidad constante la respuesta sea `0`). Por tanto, no se puede dividir por la suma.

2. **¿Cómo afecta el número de bins (`nbins`) al histograma?**
   - **Pocos bins (ej. `nbins=8`):** Agrupa muchas intensidades en cada barra; da una visión muy general pero oculta picos y detalles finos de la distribución.
   - **Muchos bins (ej. `nbins=128` o `256`):** Muestra la distribución exacta nivel por nivel con gran detalle (un bin por cada valor en `uint8` cuando `nbins=256`). Un valor intermedio (`32`-`64`) suele ofrecer un buen equilibrio entre detalle y suavidad visual.

3. **¿Qué le ocurre al histograma al reducir (`rescale 1/20`) o ampliar (`rescale 20`) una imagen?**
   - **Al reducir (Downsampling `1/20`):** Se pierde resolución y detalle al promediar píxeles vecinos. Hay muchos menos píxeles totales (frecuencias mucho más bajas) y menos valores distintos de intensidad, por lo que el histograma se vuelve **disperso (*sparse*)** y con menor rango dinámico.
   - **Al ampliar (Upsampling `20x`):** Se interpolan nuevos píxeles a partir de los vecinos. La forma general del histograma se conserva idéntica a la original, pero las cuentas de frecuencia en el eje Y son mucho mayores ($20^2 = 400$ veces más píxeles) sin añadir nuevos niveles de intensidad extremos.

4. **¿Qué efecto tiene aplicar una convolución de suavizado sobre el histograma de una imagen?**
   - El suavizado elimina las **altas frecuencias** (bordes y transiciones bruscas) promediando cada píxel con sus vecinos. Esto **reduce la varianza** de las intensidades: desaparecen los valores extremos (muy oscuros o muy claros) y las intensidades se concentran hacia los valores medios (pico central más estrecho y pronunciado).

5. **¿Qué diferencias hay entre un filtro 1D horizontal, 1D vertical y 2D?**
   - **1D Horizontal (`1×15`):** Desenfoca solo en dirección horizontal; difumina los bordes verticales pero preserva mejor los bordes horizontales.
   - **1D Vertical (`15×1`):** Desenfoca solo en dirección vertical; difumina los bordes horizontales pero preserva mejor los bordes verticales.
   - **2D (`15×15`):** Desenfoca en ambas direcciones simultáneamente (promedia 225 píxeles en lugar de 15), produciendo un desenfoque global mucho más fuerte y reduciendo más el rango de intensidades.

6. **Comparativa Filtro Gaussiano vs Filtro de Mediana (y formas `disk`, `rectangle`, `diamond`):**
   - **Gaussiano:** Filtro lineal ponderado controlado por $\sigma$. Con $\sigma=0.5$ o $\sigma=1$ preserva estructuras principales (como los cuernos del órix) reduciendo ruido leve; con $\sigma=2$ o $\sigma=3$ desenfoca en exceso y borra detalles finos.
   - **Mediana:** Filtro **no lineal** excelente para eliminar ruido impulsivo (*sal y pimienta*) preservando bordes mejor que el gaussiano. Sin embargo, si el elemento estructurante es grande (`disk(5)` o `rectangle(5, 20)`), **elimina estructuras finas y delgadas** como los cuernos del órix (el rectangular además distorsiona estirando horizontalmente). El más pequeño (`diamond(3)`) logra el mejor equilibrio conservando los cuernos.

7. **Comparativa Prewitt vs Sobel vs Canny (¿Mejora el suavizado previo?):**
   - **Prewitt y Sobel:** Calculan solo la magnitud del gradiente; devuelven una imagen en **escala de grises** con bordes gruesos/borrosos y son muy sensibles al ruido y texturas (barba, piel). Requieren umbralización posterior.
   - **Canny:** Incluye suavizado gaussiano ($\sigma$), cálculo de gradiente, **supresión de no-máximos** (adelgazamiento de bordes a 1 píxel) y **umbralización por histéresis**. Devuelve una **imagen binaria** limpia y precisa.
   - **¿Suavizar antes?** Sí: aplicar un suavizado gaussiano previo reduce el ruido de alta frecuencia y evita falsos bordes en texturas finas, aunque un suavizado excesivo puede hacer perder bordes débiles.